In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, recall_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTENC
from imblearn.ensemble import BalancedRandomForestClassifier

from xgboost import XGBClassifier


os.makedirs("images", exist_ok=True)
os.makedirs("tables", exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SIZE = 60000
classes = ["Fatal", "Serious", "Slight"]
label_map = {"Fatal": 0, "Serious": 1, "Slight": 2}
reverse_map = {0: "Fatal", 1: "Serious", 2: "Slight"}


df = pd.read_csv(
    "Accident_Information.csv",
    encoding="latin1",
    engine="python",
    on_bad_lines="skip"
)

cols = [
    "Accident_Severity", "Date", "Time", "Year", "Day_of_Week",
    "Speed_limit", "Road_Type", "1st_Road_Class", "2nd_Road_Class",
    "Weather_Conditions", "Light_Conditions", "Road_Surface_Conditions",
    "Junction_Control", "Junction_Detail",
    "Pedestrian_Crossing-Human_Control",
    "Pedestrian_Crossing-Physical_Facilities",
    "Special_Conditions_at_Site", "Carriageway_Hazards",
    "Urban_or_Rural_Area", "Number_of_Vehicles"
]

df = df[[c for c in cols if c in df.columns]].copy()
df = df.dropna(subset=["Accident_Severity"])

df["Accident_Severity"] = df["Accident_Severity"].replace({
    1: "Fatal", 2: "Serious", 3: "Slight",
    "1": "Fatal", "2": "Serious", "3": "Slight"
})
df = df[df["Accident_Severity"].isin(classes)]

if len(df) > SAMPLE_SIZE:
    df, _ = train_test_split(
        df,
        train_size=SAMPLE_SIZE,
        stratify=df["Accident_Severity"],
        random_state=RANDOM_STATE
    )


if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df["Month"] = df["Date"].dt.month

if "Time" in df.columns:
    df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

if "Day_of_Week" in df.columns:
    df["Is_Weekend"] = df["Day_of_Week"].isin([1, 7]).astype(int)

if "Hour" in df.columns:
    df["Is_Night"] = df["Hour"].apply(
        lambda h: 1 if pd.notna(h) and (h >= 20 or h <= 5) else 0
    )

if "Speed_limit" in df.columns:
    df["Speed_limit"] = pd.to_numeric(df["Speed_limit"], errors="coerce")
    df["High_Speed_Road"] = (df["Speed_limit"] >= 60).astype(int)

df = df.drop(columns=[c for c in ["Date", "Time"] if c in df.columns])


df["Accident_Severity"].value_counts().reindex(classes).plot(kind="bar")
plt.title("Accident Severity Distribution")
plt.xlabel("Severity")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("images/figure1_severity_distribution.png", dpi=300)
plt.close()

(df["Accident_Severity"].value_counts(normalize=True).reindex(classes) * 100).plot(kind="bar")
plt.title("Accident Severity Percentage")
plt.xlabel("Severity")
plt.ylabel("Percentage")
plt.tight_layout()
plt.savefig("images/figure2_severity_percentage.png", dpi=300)
plt.close()

if "Speed_limit" in df.columns:
    pd.crosstab(df["Speed_limit"], df["Accident_Severity"], normalize="index").plot(
        kind="bar", stacked=True
    )
    plt.title("Severity Proportion by Speed Limit")
    plt.tight_layout()
    plt.savefig("images/figure3_speed_limit.png", dpi=300)
    plt.close()

if "Urban_or_Rural_Area" in df.columns:
    pd.crosstab(df["Urban_or_Rural_Area"], df["Accident_Severity"], normalize="index").plot(
        kind="bar", stacked=True
    )
    plt.title("Severity Proportion by Urban or Rural Area")
    plt.tight_layout()
    plt.savefig("images/figure4_urban_rural.png", dpi=300)
    plt.close()


X = df.drop("Accident_Severity", axis=1)
y = df["Accident_Severity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

cat_cols = X_train.select_dtypes(include="object").columns.tolist()
num_cols = X_train.select_dtypes(exclude="object").columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])

cat_pipe_ohe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_ohe = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe_ohe, cat_cols)
])

cat_pipe_ord = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocess_ord = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe_ord, cat_cols)
])

X_train_ord = preprocess_ord.fit_transform(X_train)
X_test_ord = preprocess_ord.transform(X_test)
cat_indices = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

y_train_num = y_train.map(label_map)
y_test_num = y_test.map(label_map)

results = []


def score_model(name, y_true, y_pred):
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro", labels=classes, zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", labels=classes, zero_division=0),
        "Fatal Recall": recall_score(y_true, y_pred, labels=["Fatal"], average="macro", zero_division=0),
        "Serious Recall": recall_score(y_true, y_pred, labels=["Serious"], average="macro", zero_division=0)
    }
    results.append(row)

    print("\n", name)
    print(pd.Series(row))
    print(classification_report(y_true, y_pred, labels=classes, zero_division=0))


def save_cm(name, y_pred, fig_no):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, labels=classes, xticks_rotation=45
    )
    plt.title(name)
    plt.tight_layout()
    plt.savefig(f"images/figure{fig_no}_{name.lower().replace(' ', '_').replace('-', '')}.png", dpi=300)
    plt.close()


def decode(pred):
    return pd.Series(pred).map(reverse_map).values


model_list = [
    ("Logistic Regression - Imbalanced",
     Pipeline([("prep", preprocess_ohe),
               ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])),

    ("Random Forest - Imbalanced",
     Pipeline([("prep", preprocess_ohe),
               ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))])),

    ("Logistic Regression - Class Weighted",
     Pipeline([("prep", preprocess_ohe),
               ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))])),

    ("Random Forest - Class Weighted",
     Pipeline([("prep", preprocess_ohe),
               ("model", RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                                random_state=RANDOM_STATE, n_jobs=-1))])),

    ("Logistic Regression - RandomOverSampler",
     ImbPipeline([("prep", preprocess_ohe),
                  ("ros", RandomOverSampler(random_state=RANDOM_STATE)),
                  ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])),

    ("Random Forest - RandomOverSampler",
     ImbPipeline([("prep", preprocess_ohe),
                  ("ros", RandomOverSampler(random_state=RANDOM_STATE)),
                  ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))])),

    ("Balanced Random Forest",
     Pipeline([("prep", preprocess_ohe),
               ("model", BalancedRandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))]))
]

fig_no = 6

for name, model in model_list:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    score_model(name, y_test, preds)
    save_cm(name, preds, fig_no)
    fig_no += 1


xgb_base = Pipeline([
    ("prep", preprocess_ohe),
    ("model", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        eval_metric="mlogloss",
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

xgb_base.fit(X_train, y_train_num)
xgb_preds = decode(xgb_base.predict(X_test))
score_model("XGBoost - Imbalanced", y_test, xgb_preds)
save_cm("XGBoost - Imbalanced", xgb_preds, fig_no)
fig_no += 1


X_train_ohe = preprocess_ohe.fit_transform(X_train)
X_test_ohe = preprocess_ohe.transform(X_test)

ros = RandomOverSampler(random_state=RANDOM_STATE)
X_ros, y_ros = ros.fit_resample(X_train_ohe, y_train_num)

xgb_ros = XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_ros.fit(X_ros, y_ros)
xgb_ros_preds = decode(xgb_ros.predict(X_test_ohe))
score_model("XGBoost - RandomOverSampler", y_test, xgb_ros_preds)
save_cm("XGBoost - RandomOverSampler", xgb_ros_preds, fig_no)
fig_no += 1


smote = SMOTENC(categorical_features=cat_indices, random_state=RANDOM_STATE)
X_smote, y_smote = smote.fit_resample(X_train_ord, y_train)

rf_smote = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_smote.fit(X_smote, y_smote)
rf_smote_preds = rf_smote.predict(X_test_ord)
score_model("Random Forest - SMOTENC", y_test, rf_smote_preds)
save_cm("Random Forest - SMOTENC", rf_smote_preds, fig_no)
fig_no += 1

xgb_smote = XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_smote.fit(X_smote, pd.Series(y_smote).map(label_map))
xgb_smote_preds = decode(xgb_smote.predict(X_test_ord))
score_model("XGBoost - SMOTENC", y_test, xgb_smote_preds)
save_cm("XGBoost - SMOTENC", xgb_smote_preds, fig_no)
fig_no += 1


tune_pipe = ImbPipeline([
    ("prep", preprocess_ohe),
    ("ros", RandomOverSampler(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

params = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt"]
}

search = RandomizedSearchCV(
    tune_pipe,
    params,
    n_iter=5,
    scoring="f1_macro",
    cv=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
tuned_preds = best_model.predict(X_test)

score_model("Random Forest - RandomOverSampler - Tuned", y_test, tuned_preds)
save_cm("Tuned Random Forest - RandomOverSampler", tuned_preds, 20)

print("\nBest tuning parameters:")
print(search.best_params_)


before = y_train.value_counts().reindex(classes)
after_ros = pd.Series(ros.fit_resample(X_train_ohe, y_train)[1]).value_counts().reindex(classes)
after_smote = pd.Series(y_smote).value_counts().reindex(classes)

pd.DataFrame({
    "Before Balancing": before,
    "After RandomOverSampler": after_ros,
    "After SMOTENC": after_smote
}).plot(kind="bar")

plt.title("Training Class Distribution Before and After Balancing")
plt.xlabel("Severity")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("images/figure5_training_distribution_balancing.png", dpi=300)
plt.close()


results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
results_df.to_csv("tables/final_model_comparison.csv", index=False)

results_df.plot(x="Model", y="Macro F1", kind="barh", legend=False)
plt.title("Model Comparison by Macro F1-score")
plt.tight_layout()
plt.savefig("images/figure17_macro_f1.png", dpi=300)
plt.close()

results_df.plot(x="Model", y="Balanced Accuracy", kind="barh", legend=False)
plt.title("Model Comparison by Balanced Accuracy")
plt.tight_layout()
plt.savefig("images/figure18_balanced_accuracy.png", dpi=300)
plt.close()

results_df.set_index("Model")[["Fatal Recall", "Serious Recall"]].plot(kind="barh")
plt.title("Minority-Class Recall Comparison")
plt.tight_layout()
plt.savefig("images/figure19_minority_recall.png", dpi=300)
plt.close()


prep = best_model.named_steps["prep"]
rf = best_model.named_steps["model"]

try:
    feature_names = prep.get_feature_names_out()
except Exception:
    feature_names = [f"feature_{i}" for i in range(len(rf.feature_importances_))]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df.to_csv("tables/feature_importance.csv", index=False)

top20 = importance_df.head(20)
top20.plot(x="Feature", y="Importance", kind="barh", legend=False)
plt.title("Top 20 Feature Importances - Tuned Random Forest")
plt.tight_layout()
plt.savefig("images/figure21_feature_importance.png", dpi=300)
plt.close()

print("\nFinal model comparison:")
print(results_df)

print("\nProject completed. Figures are saved in the images folder.")